# Transient check: the lap in the time domain

The quasi-steady solver assumes every point is a steady state. The transient solver
(`lapsim/solvers/transient.py`, explicit RK4 at 2 ms in `core/integrator.py`) drives the same car along the
same track with pitch, roll, heave, yaw and tire lag, steered by a preview driver that follows the
quasi-steady speed profile scaled by the driver level. Where the two disagree, the steady-state assumption
is costing accuracy.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
car_pick = W.ToggleButtons(options=list(CARS), description="Car")
event = W.ToggleButtons(options=["acceleration", "skidpad", "autocross"], value="skidpad", description="Event")
level = W.FloatSlider(value=0.98, min=0.85, max=1.0, step=0.01, description="driver level", continuous_update=False)
out = W.Output()


def run():
    car = lapsim.load_car(CARS[car_pick.value])
    res = car.run((event.value,))
    lr = res.laps[event.value]
    ch = lapsim.solvers.transient.Chassis(res.vehicle, driver_level=level.value)
    r = ch.run(lr.track, lr, v0=1.0 if event.value != "skidpad" else None, name=event.value)
    print(f"quasi-steady {lr.time:.3f} s, transient {r.summary['time_s']:.3f} s, completed: {r.summary['completed']}")
    df, q = r.df, r.qss
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
    ax[0].plot(df["s"], 3.6 * df["v"], label="transient"); ax[0].plot(q["s"], 3.6 * q["v"], label="quasi-steady")
    ax[0].set(xlabel="distance [m]", ylabel="km/h", title="Speed"); ax[0].legend()
    ax[1].plot(df["s"], df["roll_deg"], label="transient"); ax[1].plot(q["s"], q["roll_deg"], label="quasi-steady")
    ax[1].set(xlabel="distance [m]", ylabel="deg", title="Roll")
    ax[2].plot(df["x"], df["y"], lw=0.8); ax[2].set(aspect="equal", title="Path driven")
    plt.show()
    for n in r.notes:
        print("note:", n)


display(W.VBox([car_pick, event, level, run_button("Run", run, out), out]))
with out:
    run()